# 10 — Wind-Calm Pollution Accumulation Test


This notebook tests whether calm conditions produce a stronger coastal-to-inland accumulation signal across Eastbourne, Newhaven, and Lewes.


In [ ]:

SITES = [
    {"site_id": "EST", "site_name": "Eastbourne",   "lat": 50.7680,  "lon": 0.2900},
    {"site_id": "NHV", "site_name": "Newhaven ERF", "lat": 50.80208, "lon": 0.04961},
    {"site_id": "LWS", "site_name": "Lewes",        "lat": 50.8739,  "lon": 0.0110},
]
DATE_FROM = "2024-12-26"
DATE_TO   = "2025-01-05"
CALM_WIND_THRESHOLD_MS = 2.0
SCENE_STATS_DIR   = "outputs/s5p_stack"
OUTPUT_DIR        = "outputs/wind_calm_test"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:

def fetch_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "wind_speed_10m,wind_direction_10m,cloud_cover",
        "timezone": "UTC",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    j = r.json()
    h = j.get("hourly", {})
    return pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "wind_dir_deg": h.get("wind_direction_10m", []),
        "cloud_cover_pct": h.get("cloud_cover", []),
    })

wx_rows = []
for site in SITES:
    w = fetch_weather(site["lat"], site["lon"], DATE_FROM, DATE_TO)
    w["site_id"] = site["site_id"]
    w["site_name"] = site["site_name"]
    w["date"] = w["time_utc"].dt.date.astype("string")
    wx_rows.append(w)
wx = pd.concat(wx_rows, ignore_index=True)
daily_wx = (
    wx.groupby(["site_id","site_name","date"], dropna=False)
    .agg(
        mean_wind_speed_ms=("wind_speed_ms","mean"),
        mean_cloud_cover_pct=("cloud_cover_pct","mean"),
        mean_wind_dir_deg=("wind_dir_deg","mean"),
    )
    .reset_index()
)
display(daily_wx.head())


In [ ]:

stats_rows = []
for site in SITES:
    p_csv = Path(SCENE_STATS_DIR) / f"{site['site_id']}_scene_level_no2_window_stats.csv"
    p_pq = Path(SCENE_STATS_DIR) / f"{site['site_id']}_scene_level_no2_window_stats.parquet"
    p = p_csv if p_csv.exists() else p_pq if p_pq.exists() else None
    if p is None:
        continue
    df = pd.read_csv(p) if p.suffix == ".csv" else pd.read_parquet(p)
    if "start_utc" in df.columns:
        df["start_utc"] = pd.to_datetime(df["start_utc"], utc=True, errors="coerce")
        df["date"] = df["start_utc"].dt.date.astype("string")
    df["site_id"] = site["site_id"]
    df["site_name"] = site["site_name"]
    stats_rows.append(df)
stats = pd.concat(stats_rows, ignore_index=True) if stats_rows else pd.DataFrame()
print("Stats rows:", len(stats))
display(stats.head())


In [ ]:

calm_days = daily_wx[daily_wx["mean_wind_speed_ms"].fillna(99) < CALM_WIND_THRESHOLD_MS].copy()
calm_days["calm_flag"] = True

if not stats.empty and "mean_no2" in stats.columns:
    stats_daily = (
        stats.groupby(["site_id","site_name","date"], dropna=False)
        .agg(
            mean_no2=("mean_no2","mean"),
            median_no2=("median_no2","mean"),
        )
        .reset_index()
    )
    calm_days = calm_days.merge(stats_daily, on=["site_id","site_name","date"], how="left")

pivot = calm_days.pivot_table(index="date", columns="site_id", values="mean_no2", aggfunc="mean")
if not pivot.empty and {"EST","LWS"}.issubset(set(pivot.columns)):
    pivot["LWS_minus_EST"] = pivot["LWS"] - pivot["EST"]
if not pivot.empty and {"NHV","EST"}.issubset(set(pivot.columns)):
    pivot["NHV_minus_EST"] = pivot["NHV"] - pivot["EST"]

calm_csv = Path(OUTPUT_DIR) / "wind_calm_candidate_days.csv"
pivot_csv = Path(OUTPUT_DIR) / "wind_calm_gradients.csv"
calm_days.to_csv(calm_csv, index=False)
pivot.reset_index().to_csv(pivot_csv, index=False)
display(calm_days.head(20))
display(pivot.reset_index().head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
if not pivot.empty:
    for col in ["LWS_minus_EST", "NHV_minus_EST"]:
        if col in pivot.columns:
            ax.plot(pd.to_datetime(pivot.index), pivot[col], marker="o", label=col)
ax.axhline(0, linestyle="--")
ax.set_title("Wind-calm inland-minus-coastal NO2 difference")
ax.set_xlabel("Date")
ax.set_ylabel("Difference")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / "wind_calm_difference_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "wind_calm_pollution_accumulation",
    "sites": SITES,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "calm_wind_threshold_ms": CALM_WIND_THRESHOLD_MS,
    "rows_weather": int(len(wx)),
    "rows_stats": int(len(stats)),
    "rows_calm_days": int(len(calm_days)),
    "outputs": {
        "calm_days_csv": str(calm_csv),
        "gradient_csv": str(pivot_csv),
        "plot_png": str(plot_path),
    },
}
manifest_path = Path(OUTPUT_DIR) / "wind_calm_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", calm_csv)
print("Saved:", pivot_csv)
print("Saved:", plot_path)
print("Saved:", manifest_path)
